# FinSearch: Finance-Aware Chatbot
**Pipeline:** Hybrid Retrieval -> Groq LLM Re-ranking -> Answer Generation -> NLI Confidence Score

In [1]:
import sys, os
from pathlib import Path

_root = Path().resolve()
for _ in range(5):
    if (_root / 'config.py').exists():
        break
    _root = _root.parent

os.chdir(_root)
sys.path.insert(0, str(_root))

from config import FIQA_CORPUS, FIQA_QUERIES, FIQA_QRELS_TEST

DENSE_INDEX_DIR = os.path.join('dense_retrieval', 'Results', 'indexes')
FINRERANK_DIR   = os.path.join('finrerank', 'Results')

print('Repo root :', _root)
print('Index dir :', DENSE_INDEX_DIR)


Repo root : D:\GEN AI Project\FinSearch_Intent_Aware_Financial_Document_Intelligence-
Index dir : dense_retrieval\Results\indexes


In [2]:
# Uncomment if not installed
# !pip install openai sentence-transformers faiss-cpu rank_bm25 pytrec_eval --user


In [3]:
import os
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"

import pandas as pd
import numpy as np
import json, re, time, pickle
import pytrec_eval
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from openai import OpenAI
from tqdm import tqdm

print("Imports OK")


c:\Users\Kiruba\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


In [4]:
print('Loading corpus...')
corpus_df = pd.read_csv(FIQA_CORPUS, usecols=['_id', 'text'])
corpus_df = corpus_df.dropna(subset=['text']).reset_index(drop=True)
corpus_df['_id'] = corpus_df['_id'].astype(str)
corpus_id_to_text = dict(zip(corpus_df['_id'], corpus_df['text']))
print('Corpus:', len(corpus_id_to_text), 'passages')

queries_df = pd.read_csv(FIQA_QUERIES, usecols=['_id', 'text'])
queries_df['_id'] = queries_df['_id'].astype(str)
query_id_to_text = dict(zip(queries_df['_id'], queries_df['text']))

qrels_df = pd.read_csv(FIQA_QRELS_TEST)
qrels_df.columns = [c.strip() for c in qrels_df.columns]
qrels_df = qrels_df.astype(str)
query_col  = [c for c in qrels_df.columns if 'query'  in c.lower()][0]
corpus_col = [c for c in qrels_df.columns if 'corpus' in c.lower()][0]
score_col  = [c for c in qrels_df.columns if 'score'  in c.lower()][0]
qrels_dict = {}
for _, row in qrels_df.iterrows():
    qid = str(row[query_col])
    did = str(row[corpus_col])
    if qid not in qrels_dict:
        qrels_dict[qid] = {}
    qrels_dict[qid][did] = int(float(row[score_col]))
test_qids = list(qrels_dict.keys())
test_queries_df = queries_df[queries_df['_id'].isin(test_qids)].reset_index(drop=True)
print('Test queries:', len(test_queries_df), '| Qrels:', len(qrels_dict))


Loading corpus...
Corpus: 57600 passages
Test queries: 648 | Qrels: 648


In [5]:
faiss_path = os.path.join(DENSE_INDEX_DIR, 'faiss_minilm.index')
ids_path   = os.path.join(DENSE_INDEX_DIR, 'corpus_ids.pkl')
index      = faiss.read_index(faiss_path)
with open(ids_path, 'rb') as f:
    corpus_ids = pickle.load(f)
print('FAISS index:', index.ntotal, 'vectors')

dense_model = SentenceTransformer('all-MiniLM-L6-v2')
print('Dense model loaded')

def tokenize(text):
    t = str(text).lower()
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    return t.split()

print('Building BM25 (1-2 min)...')
tokenized_corpus = [tokenize(t) for t in tqdm(corpus_df['text'])]
bm25 = BM25Okapi(tokenized_corpus)
print('BM25 ready')


FAISS index: 57600 vectors
Dense model loaded
Building BM25 (1-2 min)...


100%|██████████| 57600/57600 [00:03<00:00, 18209.13it/s]


BM25 ready


In [7]:
ALPHA = 0.7
TOP_K = 20

print('Encoding queries...')
query_texts = test_queries_df['text'].tolist()
query_ids   = test_queries_df['_id'].tolist()
query_embs  = dense_model.encode(query_texts, normalize_embeddings=True,
                                  batch_size=64, show_progress_bar=True)

print('Running Hybrid retrieval (top-20)...')
hybrid_top20 = {}
for i, (qid, q_text) in enumerate(tqdm(zip(query_ids, query_texts), total=len(query_ids))):
    q_vec = query_embs[i].reshape(1, -1).astype('float32')
    D, I  = index.search(q_vec, TOP_K * 5)
    dense_scores = {corpus_ids[idx]: float(D[0][j]) for j, idx in enumerate(I[0]) if idx < len(corpus_ids)}
    toks         = tokenize(q_text)
    bm25_raw     = bm25.get_scores(toks)
    top_bm25_idx = np.argsort(bm25_raw)[::-1][:TOP_K * 5]
    bm25_scores  = {str(corpus_df.iloc[idx]['_id']): float(bm25_raw[idx]) for idx in top_bm25_idx}
    all_docs = set(dense_scores) | set(bm25_scores)
    d_vals   = np.array([dense_scores.get(d, 0.0) for d in all_docs])
    b_vals   = np.array([bm25_scores.get(d,  0.0) for d in all_docs])
    d_norm   = (d_vals - d_vals.min()) / (d_vals.max() - d_vals.min() + 1e-9)
    b_norm   = (b_vals - b_vals.min()) / (b_vals.max() - b_vals.min() + 1e-9)
    hybrid   = ALPHA * d_norm + (1 - ALPHA) * b_norm
    ranked   = sorted(zip(all_docs, hybrid), key=lambda x: x[1], reverse=True)[:TOP_K]
    hybrid_top20[qid] = {doc_id: float(score) for doc_id, score in ranked}
print('Hybrid done:', len(hybrid_top20), 'queries')


Encoding queries...


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Batches: 100%|██████████| 11/11 [00:03<00:00,  3.61it/s]


Running Hybrid retrieval (top-20)...


100%|██████████| 648/648 [02:42<00:00,  4.00it/s]

Hybrid done: 648 queries


In [8]:
OPENROUTER_API_KEY = "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"   # paste your OpenRouter key here

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

MODEL = "meta-llama/llama-3.3-70b-instruct"  # same model as before

# Smoke test
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with the single word: ready"}],
    max_tokens=5,
)
print("OpenRouter response:", resp.choices[0].message.content.strip())
print("OpenRouter + Llama 3.3 70B ready")


OpenRouter response: ready
OpenRouter + Llama 3.3 70B ready


In [9]:
def groq_rerank(query, candidates, corpus_id_to_text, client, top_k=5):
    lines = []
    for i, d in enumerate(candidates, 1):
        text = corpus_id_to_text.get(d, "")[:400].replace("\n", " ")
        lines.append(str(i) + ". " + text)
    n = len(candidates)
    passages = "\n".join(lines)
    prompt = (
        "You are a financial information retrieval expert.\n\n"
        "Query: \"" + query + "\"\n\n"
        "Below are " + str(n) + " candidate passages. "
        "Return a JSON array of passage numbers (1-based) sorted MOST to LEAST relevant. "
        "Include ALL " + str(n) + " numbers.\n\n"
        "Passages:\n" + passages + "\n\n"
        "Respond ONLY with a JSON array, e.g.: [3,1,5,2,4]"
    )
    try:
        r = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=150, temperature=0,
        )
        raw = r.choices[0].message.content.strip()
        m = re.search(r"\\[[\\d,\\s]+\\]", raw)
        if m:
            ranking = json.loads(m.group())
            ranking = [x for x in ranking if 1 <= x <= n]
            seen = set(ranking)
            for i in range(1, n+1):
                if i not in seen:
                    ranking.append(i)
            return {candidates[idx-1]: float(n - pos + 1) for pos, idx in enumerate(ranking[:top_k], 1)}
    except Exception:
        pass
    return {d: float(n - i) for i, d in enumerate(candidates[:top_k])}

print("groq_rerank() defined")


groq_rerank() defined


In [10]:
def groq_answer(query, top_docs, corpus_id_to_text, client):
    parts = []
    for i, d in enumerate(top_docs, 1):
        parts.append("[Doc " + str(i) + "] " + corpus_id_to_text.get(d, "")[:500])
    context = "\n".join(parts)
    prompt = (
        "You are an expert financial advisor assistant.\n"
        "Answer the question using ONLY the provided documents.\n"
        "If the documents do not contain enough information, say so.\n\n"
        "Question: " + query + "\n\n"
        "Documents:\n" + context + "\n\n"
        "Provide a clear, accurate financial answer based strictly on the documents above."
    )
    try:
        r = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=300, temperature=0.1,
        )
        return r.choices[0].message.content.strip()
    except Exception as e:
        return "Error: " + str(e)

print("groq_answer() defined")


groq_answer() defined


In [11]:
print("Loading NLI confidence model...")
nli_model = CrossEncoder("cross-encoder/nli-deberta-v3-small")
print("NLI model loaded")

def compute_confidence(answer, top_docs, corpus_id_to_text, hybrid_scores, nli_model):
    # Score 1: Retrieval confidence
    scores = [hybrid_scores.get(d, 0.0) for d in top_docs]
    s_min, s_max = min(scores), max(scores)
    norm = [(s - s_min) / (s_max - s_min + 1e-9) for s in scores]
    retrieval_conf = float(np.mean(norm))

    # Score 2: Faithfulness via NLI entailment
    # Does each doc SUPPORT the generated answer?
    # NLI label order: [contradiction=0, entailment=1, neutral=2]
    entailment_scores = []
    for doc_id in top_docs:
        doc_text = corpus_id_to_text.get(doc_id, "")[:500]
        if not doc_text.strip():
            continue
        logits = nli_model.predict([[doc_text, answer]])
        probs  = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
        entailment_scores.append(float(probs[0][1]))
    faithfulness_conf = float(np.mean(entailment_scores)) if entailment_scores else 0.0

    final_conf = round(0.4 * retrieval_conf + 0.6 * faithfulness_conf, 4)
    return {
        "retrieval_confidence":    round(retrieval_conf,    4),
        "faithfulness_confidence": round(faithfulness_conf, 4),
        "final_confidence":        final_conf
    }

print("compute_confidence() defined")


Loading NLI confidence model...
NLI model loaded
compute_confidence() defined


In [12]:
# Demo on 5 sample queries
SAMPLE_QIDS = test_queries_df["_id"].head(5).tolist()
RATE_DELAY  = 2.5
demo_results = []

for qid in SAMPLE_QIDS:
    query      = query_id_to_text.get(qid, "")
    candidates = list(hybrid_top20[qid].keys())

    # Step 1: Groq re-ranks 20 -> top-5
    reranked  = groq_rerank(query, candidates, corpus_id_to_text, client, top_k=5)
    top5_docs = list(reranked.keys())
    time.sleep(RATE_DELAY)

    # Step 2: Groq generates financial answer
    answer = groq_answer(query, top5_docs, corpus_id_to_text, client)
    time.sleep(RATE_DELAY)

    # Step 3: True confidence score via NLI
    conf  = compute_confidence(answer, top5_docs, corpus_id_to_text, hybrid_top20[qid], nli_model)
    fc    = conf["final_confidence"]
    label = "HIGH - Answer confidently" if fc >= 0.7 else ("MEDIUM - Answer with caveat" if fc >= 0.4 else "LOW - Cannot answer reliably")

    demo_results.append({"query_id": qid, "query": query, "answer": answer,
                         "top5_docs": top5_docs, **conf, "conf_label": label})

    print("=" * 70)
    print("Query           :", query)
    print("Answer          :", answer[:300])
    print("Retrieval Conf  :", conf["retrieval_confidence"])
    print("Faithfulness    :", conf["faithfulness_confidence"])
    print("Final Confidence:", fc, "-->", label)
    print()


C:\Users\Kiruba\AppData\Local\Temp\ipykernel_32240\3807273619.py:24: FutureWarning: Possible nested set at position 3
  m = re.search(r"\\[[\\d,\\s]+\\]", raw)


Query           : Where should I park my rainy-day / emergency fund?
Answer          : Based on the provided documents, for a rainy-day/emergency fund, you should consider parking your money in:

1. A savings account
2. A money market account
3. CDs (with a definite time period, such as 6 months, 1 year, or 2 years)
4. A highly liquid exchange-traded product with low volatility/drawdo
Retrieval Conf  : 0.3561
Faithfulness    : 0.1962
Final Confidence: 0.2602 --> LOW - Cannot answer reliably

Query           : Tax considerations for selling a property below appraised value to family?
Answer          : Based on the provided documents, selling a property below appraised value to a family member may result in trouble deducting losses on taxes, especially since it's a related person transaction. The state of Maryland has a transfer/recordation tax of 1.5% for both the buyer and seller, but it's uncle
Retrieval Conf  : 0.5365
Faithfulness    : 0.003
Final Confidence: 0.2164 --> LOW - Cannot 

In [ ]:
# Full evaluation on all 648 queries (~30 min, optional)
# Skip this cell if you only need the demo above

all_reranked = {}
for _, row in tqdm(test_queries_df.iterrows(), total=len(test_queries_df)):
    qid      = str(row["_id"])
    reranked = groq_rerank(str(row["text"]), list(hybrid_top20[qid].keys()),
                           corpus_id_to_text, client, top_k=10)
    all_reranked[qid] = reranked
    time.sleep(2.5)

evaluator    = pytrec_eval.RelevanceEvaluator(qrels_dict, {"ndcg_cut_10", "recip_rank", "recall_10"})
eval_results = evaluator.evaluate(all_reranked)
metrics = {
    "NDCG@10":   round(np.mean([v["ndcg_cut_10"] for v in eval_results.values()]), 4),
    "MRR":       round(np.mean([v["recip_rank"]  for v in eval_results.values()]), 4),
    "Recall@10": round(np.mean([v["recall_10"]   for v in eval_results.values()]), 4),
}

print("=" * 60)
comparisons = [
    ("BM25 Baseline",        0.2169, 0.2706, 0.2784),
    ("MiniLM-Dense",         0.3687, 0.4451, 0.4413),
    ("Hybrid-Alpha (a=0.7)", 0.3791, 0.4606, 0.4473),
    ("LLM Re-ranker (Groq)", metrics["NDCG@10"], metrics["MRR"], metrics["Recall@10"]),
]
print(f"{\"Model\":<28} {\"NDCG@10\":>8} {\"MRR\":>8} {\"Recall@10\":>10}")
print("-" * 60)
for name, ndcg, mrr, recall in comparisons:
    tag = "  <-- NEW" if "Groq" in name else ""
    print(f"{name:<28} {ndcg:>8.4f} {mrr:>8.4f} {recall:>10.4f}{tag}")
print("=" * 60)


In [ ]:
os.makedirs(FINRERANK_DIR, exist_ok=True)

demo_path = os.path.join(FINRERANK_DIR, "demo_results.json")
with open(demo_path, "w") as f:
    json.dump(demo_results, f, indent=2)
print("Saved:", demo_path)

try:
    metrics_out = {
        "model":       "LLM Re-ranker (Hybrid-Alpha -> Groq Llama-3.3-70B)",
        "dataset":     "FiQA Test",
        "num_queries": len(eval_results),
        **metrics
    }
    metrics_path = os.path.join(FINRERANK_DIR, "finrerank_metrics.json")
    with open(metrics_path, "w") as f:
        json.dump(metrics_out, f, indent=2)
    print("Saved:", metrics_path)
except NameError:
    print("Full metrics not saved (cell-12 was skipped)")

print("Done!")
